In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [2]:
import numpy as np
from loaders._gen_binary import generate_data
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import balanced_accuracy_score

In [3]:
_ = generate_data(n_samples=1000, verbose=True)

Generated dataset with 995 samples.
Shapes: [(796, 25), (0, 25), (199, 25)]


In [4]:
bacc = []

for seed in range(30):
    pack = generate_data(seed=seed)
    X_train, y_train = pack["train"]
    X_test, y_test = pack["test"]

    tscv = TimeSeriesSplit(n_splits=5)

    param_dist = {
        "n_estimators": [10, 20, 50, 100],
        "max_depth": [3, 5, 7, 9],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    }

    search = RandomizedSearchCV(
        estimator=RandomForestClassifier(),
        param_distributions=param_dist,
        n_iter=5,        
        cv=tscv,             
        n_jobs=-1,
        random_state=42
    )

    search.fit(X_train, y_train)
    y_preds = search.predict(X_test)

    bacc.append(balanced_accuracy_score(y_test, y_preds))

print(f"Mean Test Balanced Accuracy: {np.mean(bacc)}")
print(f"Std Test Balanced Accuracy: {np.std(bacc)}")

Mean Test Balanced Accuracy: 0.685671507898116
Std Test Balanced Accuracy: 0.02078976181642646
